# Lecture 4 Colab: Financial News Sentiment with TF-IDF

This notebook demonstrates a traditional NLP pipeline for financial news sentiment classification using a small synthetic headline dataset.


## Learning objectives

- Understand a simple NLP classification pipeline.
- Convert financial headlines into numerical TF-IDF features.
- Train a Naive Bayes or Logistic Regression classifier.
- Evaluate classification results.
- Understand why sentiment labels may be subjective.


## Connection to Lecture 4

Supports NLP, sentiment analysis, text preprocessing, word representation, Naive Bayes, and logistic regression.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
np.random.seed(42)


## Dataset explanation

The headlines are synthetic teaching examples. They are not downloaded from news providers and should not be treated as market evidence.


In [ ]:
rows = [{'headline': 'Bank reports stronger quarterly profit', 'sentiment': 'positive'}, {'headline': 'Shares rise after better than expected earnings', 'sentiment': 'positive'}, {'headline': 'Company raises full year revenue guidance', 'sentiment': 'positive'}, {'headline': 'Fintech firm expands digital payment partnership', 'sentiment': 'positive'}, {'headline': 'Insurer posts improved underwriting profit', 'sentiment': 'positive'}, {'headline': 'Lender announces higher net interest income', 'sentiment': 'positive'}, {'headline': 'Payments company signs major merchant agreement', 'sentiment': 'positive'}, {'headline': 'Asset manager attracts record fund inflows', 'sentiment': 'positive'}, {'headline': 'Brokerage upgrades firm after margin improvement', 'sentiment': 'positive'}, {'headline': 'Exchange operator reports higher trading revenue', 'sentiment': 'positive'}, {'headline': 'Company beats analyst profit expectations', 'sentiment': 'positive'}, {'headline': 'Digital bank reaches new customer growth milestone', 'sentiment': 'positive'}, {'headline': 'Credit card issuer reports lower delinquency rate', 'sentiment': 'positive'}, {'headline': 'Wealth platform expands assets under management', 'sentiment': 'positive'}, {'headline': 'Bank shares gain after capital ratio improves', 'sentiment': 'positive'}, {'headline': 'Firm announces successful debt refinancing', 'sentiment': 'positive'}, {'headline': 'Loan book growth supports stronger revenue', 'sentiment': 'positive'}, {'headline': 'Analysts raise target price after positive update', 'sentiment': 'positive'}, {'headline': 'Insurance group benefits from lower claims ratio', 'sentiment': 'positive'}, {'headline': 'Financial software provider wins large bank contract', 'sentiment': 'positive'}, {'headline': 'Shares fall after weak sales forecast', 'sentiment': 'negative'}, {'headline': 'Bank faces higher loan loss provisions', 'sentiment': 'negative'}, {'headline': 'Company warns of slower demand', 'sentiment': 'negative'}, {'headline': 'Regulator fines firm over compliance failures', 'sentiment': 'negative'}, {'headline': 'Credit losses increase amid economic slowdown', 'sentiment': 'negative'}, {'headline': 'Insurer reports larger catastrophe claims', 'sentiment': 'negative'}, {'headline': 'Fintech firm delays product launch after outage', 'sentiment': 'negative'}, {'headline': 'Lender cuts profit outlook on rising defaults', 'sentiment': 'negative'}, {'headline': 'Asset manager suffers heavy fund outflows', 'sentiment': 'negative'}, {'headline': 'Payment processor faces cybersecurity investigation', 'sentiment': 'negative'}, {'headline': 'Company misses earnings expectations', 'sentiment': 'negative'}, {'headline': 'Bank reports decline in fee income', 'sentiment': 'negative'}, {'headline': 'Rating agency downgrades financial institution', 'sentiment': 'negative'}, {'headline': 'Brokerage shares slide after trading volumes fall', 'sentiment': 'negative'}, {'headline': 'Firm warns inflation pressure will reduce margins', 'sentiment': 'negative'}, {'headline': 'Credit card charge offs rise sharply', 'sentiment': 'negative'}, {'headline': 'Compliance failures lead to customer remediation costs', 'sentiment': 'negative'}, {'headline': 'Digital wallet service disrupted by technical issue', 'sentiment': 'negative'}, {'headline': 'Loan approval delays trigger customer complaints', 'sentiment': 'negative'}, {'headline': 'Company suspends dividend after weak quarter', 'sentiment': 'negative'}, {'headline': 'Central bank keeps interest rate unchanged', 'sentiment': 'neutral'}, {'headline': 'Company announces annual general meeting date', 'sentiment': 'neutral'}, {'headline': 'Bank appoints new chief financial officer', 'sentiment': 'neutral'}, {'headline': 'Exchange publishes monthly trading statistics', 'sentiment': 'neutral'}, {'headline': 'Firm releases updated sustainability report', 'sentiment': 'neutral'}, {'headline': 'Insurer schedules investor presentation next week', 'sentiment': 'neutral'}, {'headline': 'Lender opens new regional branch', 'sentiment': 'neutral'}, {'headline': 'Payments firm updates mobile app terms', 'sentiment': 'neutral'}, {'headline': 'Regulator publishes consultation paper on disclosure', 'sentiment': 'neutral'}, {'headline': 'Company files routine quarterly statement', 'sentiment': 'neutral'}, {'headline': 'Bank confirms date for earnings release', 'sentiment': 'neutral'}, {'headline': 'Asset manager announces board committee changes', 'sentiment': 'neutral'}, {'headline': 'Financial group launches internal technology review', 'sentiment': 'neutral'}, {'headline': 'Central bank releases policy meeting minutes', 'sentiment': 'neutral'}, {'headline': 'Company completes previously announced name change', 'sentiment': 'neutral'}, {'headline': 'Brokerage publishes market holiday schedule', 'sentiment': 'neutral'}, {'headline': 'Firm appoints external auditor for next fiscal year', 'sentiment': 'neutral'}, {'headline': 'Exchange announces trading hours for public holiday', 'sentiment': 'neutral'}, {'headline': 'Bank publishes annual report on corporate website', 'sentiment': 'neutral'}, {'headline': 'Company hosts investor day presentation', 'sentiment': 'neutral'}]
df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
df.head()


## Display dataset

The labels are reasonably balanced across positive, neutral, and negative classes.


In [ ]:
display(df.head(10))
print(df["sentiment"].value_counts())
df["sentiment"].value_counts().plot(kind="bar", color="steelblue")
plt.title("Headline sentiment label distribution")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()


## Text to numbers

ML models cannot directly read raw text. TF-IDF converts text into numerical features and gives higher weight to terms that are informative but not too common.


In [ ]:
X = df["headline"]
y = df["sentiment"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Training size:", len(X_train))
print("Test size:", len(X_test))


## Pipelines

The pipeline combines TF-IDF and the classifier so preprocessing is applied consistently.


In [ ]:
log_reg_pipeline = Pipeline([("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)), ("clf", LogisticRegression(max_iter=1000))])
nb_pipeline = Pipeline([("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)), ("clf", MultinomialNB())])
log_reg_pipeline.fit(X_train, y_train)
nb_pipeline.fit(X_train, y_train)


## Evaluate models

Because the dataset is small, focus on the workflow rather than exact scores.


In [ ]:
for name, model in [("Logistic Regression", log_reg_pipeline), ("Multinomial Naive Bayes", nb_pipeline)]:
    print()
    print(name)
    preds = model.predict(X_test)
    print(classification_report(y_test, preds, zero_division=0))
    cm = confusion_matrix(y_test, preds, labels=model.classes_)
    ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot(cmap="Blues", values_format="d")
    plt.title(name)
    plt.show()


## Inspect vocabulary and top words

This gives a simple view of which n-grams support each class in logistic regression.


In [ ]:
vectorizer = log_reg_pipeline.named_steps["tfidf"]
clf = log_reg_pipeline.named_steps["clf"]
feature_names = np.array(vectorizer.get_feature_names_out())
for class_index, class_name in enumerate(clf.classes_):
    top_idx = np.argsort(clf.coef_[class_index])[-8:][::-1]
    print()
    print(f"Top n-grams for class {class_name!r}:")
    print(feature_names[top_idx])


## Prediction examples

Use the trained pipeline on new synthetic headlines.


In [ ]:
new_headlines = ["Bank profit rises as digital payments grow", "Company faces investigation over accounting issue", "Central bank releases policy statement", "Shares drop after management cuts outlook"]
predictions = log_reg_pipeline.predict(new_headlines)
pd.DataFrame({"headline": new_headlines, "predicted_sentiment": predictions})


## Optional comparison: unigram vs unigram + bigram

Bigrams can capture phrases such as `loan loss` or `interest rate`, but they increase the feature space.


In [ ]:
for ngram_range in [(1, 1), (1, 2)]:
    model = Pipeline([("tfidf", TfidfVectorizer(lowercase=True, ngram_range=ngram_range, min_df=1)), ("clf", LogisticRegression(max_iter=1000))])
    model.fit(X_train, y_train)
    print(f"ngram_range={ngram_range}, test accuracy={model.score(X_test, y_test):.3f}")


## Caution in finance

Positive news does not guarantee a stock price will rise. Negative news may already be priced in. Sentiment labels can be subjective, and financial NLP systems should be evaluated carefully.


## What to observe

- TF-IDF turns text into numeric inputs.
- Logistic regression and Naive Bayes are useful traditional NLP baselines.
- Small datasets can produce unstable results.


## Common pitfalls

- Too small dataset.
- Subjective labels.
- Data leakage.
- Treating sentiment as a trading signal without validation.
- Over-cleaning text for Transformer models.


## Try it yourself

- Add 10 more headlines.
- Change `ngram_range` from `(1, 1)` to `(1, 2)`.
- Compare Logistic Regression and Naive Bayes.
- Find one headline that the model misclassifies and explain why.


## Reflection questions

- Why do we need TF-IDF?
- Why might "interest rate unchanged" be neutral in one context but important in another?
- What extra information would be needed before using sentiment for investment decisions?


## Final summary

NLP classification converts text to numerical features. TF-IDF is a useful traditional baseline. Financial sentiment must be interpreted carefully.
